# Chemical similarity: fixed top-pair dumbbells + top-1k Jaccard overlap

This notebook computes dummbbell plots and Jaccard overlap matrices for molecular fingerprints and learned embeddings. The pipeline is as follows:

- Load the Biogen endpoint table and embedding arrays.
- Compute ECFP4, Atom Pair and Topological Torsion Tanimoto similarity matrices with RDKit.
- Computes an RDKit physicochemical descriptor vector (PDV), standardizes it, and computes a pairwise cosine similarity matrix.
- Computes cosine-similarity matrices for embedding representations
- Selects the **top 100,000, 10,000 and 1,000 molecular pairs** by similarity for each endpoint and representation.
- Plots fixed-pair dumbbell summaries of median absolute endpoint difference.
- Computes Jaccard similarity between the top-1,000 pair sets across representations;
- Makes a combined figure containing the dumbbell summaries and top-1k Jaccard heatmaps.


The only section you should need to edit is the file paths section below. Note that this notebook excludes **MiniMol and MolE** from all displayed analyses.

## Imports

In [13]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## File paths

In [ ]:
# -----------------------------------------------------------------------------
# EDIT THESE PATHS
# -----------------------------------------------------------------------------
DATA_CSV           = Path('/path/to/biogen.csv')
EMBEDDING_DIR      = Path("/path/to/Biogen embeddings")
OUTPUT_DIR         = Path("qsar_figures")
TANIMOTO_CACHE_DIR = Path("similarity_cache")

MOLBERT_INVALID = 'path/to/MolBERT Biogen invalid.npy'
COATI_INVALID = 'path/to/COATI Biogen invalid.npy'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TANIMOTO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [15]:
# If True, recompute the three RDKit Tanimoto matrices even when cache files exist.
FORCE_RECOMPUTE_TANIMOTO = True

SMILES_COL = "SMILES"
ENDPOINTS  = [
    "LOG HLM_CLint (mL/min/kg)",
    "LOG MDR1-MDCK ER (B-A/A-B)",
    "LOG SOLUBILITY PH 6.8 (ug/mL)",
    "LOG RLM_CLint (mL/min/kg)",
]

ENDPOINT_LABELS = {
    "LOG HLM_CLint (mL/min/kg)": "Human liver microsomal clearance",
    "LOG MDR1-MDCK ER (B-A/A-B)": "MDR1–MDCK efflux ratio",
    "LOG SOLUBILITY PH 6.8 (ug/mL)": "Solubility at pH 6.8",
    "LOG RLM_CLint (mL/min/kg)": "Rat liver microsomal clearance",
}

TOP_PAIR_COUNTS = (100_000, 10_000, 1_000)
RANDOM_STATE    = 42

## Load and align the data

In [ ]:
def clean_representation_name(name):
    """ Remove common Biogen embedding suffixes from display labels. """
    return re.sub(
        r"[\s_-]*biogen[\s_-]*embeddings?$",
        "",
        str(name),
        flags=re.IGNORECASE,
    ).strip()


def is_excluded_representation(name):
    """ MiniMol and MolE are intentionally excluded from displayed analyses. """
    cleaned = clean_representation_name(name)
    compact = re.sub(r"[^a-z0-9]", "", str(cleaned).lower())
    return ("minimol" in compact) or (compact == "mole") 


def load_biogen_table(path):
    """ Load Biogen ADMET data """
    df      = pd.read_csv(path)
    needed  = [SMILES_COL, *ENDPOINTS]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required CSV columns: {missing}")
    return df[needed].copy()


biogen = load_biogen_table(DATA_CSV)

# Drop rows containing molecules not parseable by MolBERT or COATI
molbert_invalid = np.load(MOLBERT_INVALID)
coati_invalid   = np.load(COATI_INVALID)
invalid         = (molbert_invalid | coati_invalid).astype(bool)

biogen = biogen.loc[~invalid]
biogen.head()

,SMILES,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG RLM_CLint (mL/min/kg)
0,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,0.675687,1.493167,0.089905,1.392169
1,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,0.675687,1.040780,0.550228,1.027920
2,CN(c1ncc(F)cn1)[C@H]1CCCNC1,0.675687,-0.358806,NaN,1.027920
3,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,0.675687,1.026662,1.657056,1.027920
4,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,0.996380,1.010597,NaN,1.629093


## Tanimoto matrices (Morgan, Atom Pair, Torsion)

In [17]:
def tanimoto_matrix(fps):
    """ Return a symmetric all-vs-all Tanimoto matrix."""
    n = len(fps)
    matrix = np.eye(n, dtype=np.float32)
    for i in range(n - 1):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:])
        matrix[i, i + 1:] = sims
        matrix[i + 1:, i] = sims
    return matrix


def build_rdkit_fingerprints(smiles):
    """ Fingerprint using ECFP4, atom pair and topological torsion generators. """
    mols = [Chem.MolFromSmiles(str(s)) if pd.notna(s) else None for s in smiles]

    generators = {
        "ECFP4": rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024),
        "Atom Pair": rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024),
        "Torsion": rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024),
    }
    return {
        name: [generator.GetFingerprint(mol) for mol in mols]
        for name, generator in generators.items()
    }


def load_or_compute_tanimoto_matrices(df, force=False):
    """ Load or recompute Tanimoto matrices. Allows for changing
        the fingerprint params. """
    cache_files = {
        "ECFP4":     TANIMOTO_CACHE_DIR / "morgan_tanimoto.npy",
        "Atom Pair": TANIMOTO_CACHE_DIR / "atom_pair_tanimoto.npy",
        "Torsion":   TANIMOTO_CACHE_DIR / "torsion_tanimoto.npy",
    }

    expected_shape = (len(df), len(df))
    matrices = {}
    missing  = []

    for name, path in cache_files.items():
        if path.exists() and not force:
            arr = np.load(path, mmap_mode=None)
            if arr.shape == expected_shape:
                matrices[name] = np.asarray(arr, dtype=np.float32)
                print(f"Loaded cached {name}: {path}")
            else:
                print(f"Ignoring {path}: shape {arr.shape} != {expected_shape}")
                missing.append(name)
        else:
            missing.append(name)

    if missing:
        fps = build_rdkit_fingerprints(df[SMILES_COL])
        for name in missing:
            print(f"Computing {name} Tanimoto matrix ...")
            matrices[name] = tanimoto_matrix(fps[name])
            np.save(cache_files[name], matrices[name])
            print(f"Saved {cache_files[name]}")

    return matrices


tan_mats = load_or_compute_tanimoto_matrices(
    biogen,
    force=FORCE_RECOMPUTE_TANIMOTO,
)
{key: value.shape for key, value in tan_mats.items()}

Computing ECFP4 Tanimoto matrix ...
Saved similarity_cache/morgan_tanimoto.npy
Computing Atom Pair Tanimoto matrix ...
Saved similarity_cache/atom_pair_tanimoto.npy
Computing Torsion Tanimoto matrix ...
Saved similarity_cache/torsion_tanimoto.npy


{'ECFP4': (3515, 3515), 'Atom Pair': (3515, 3515), 'Torsion': (3515, 3515)}

## RDKit PDV + embedding cosine matrices


In [18]:
def compute_rdkit_pdv_cosine(df):
    """
    Compute the RDKit physicochemical descriptor vector (PDV) for every
    molecule,  standardize the  descriptors, and return the all-vs-all 
    cosine similarity matrix.

    The output is aligned one-to-one with the rows of `df`.
    """
    mols = [
        Chem.MolFromSmiles(str(s)) if pd.notna(s) else None
        for s in df[SMILES_COL]
    ]

    descriptor_names = [name for name, _ in Descriptors.descList]
    calculator       = MolecularDescriptorCalculator(descriptor_names)

    pdv_features = np.asarray(
        [calculator.CalcDescriptors(mol) for mol in mols],
        dtype=np.float64,
    )

    pdv_scaled = StandardScaler().fit_transform(
        VarianceThreshold().fit_transform(pdv_features)
    )

    pdv_similarity = cosine_similarity(pdv_scaled).astype(np.float32)

    print(
        f"Computed RDKit PDV cosine matrix: {pdv_similarity.shape} "
        f"from {pdv_scaled.shape[1]} retained descriptors"
    )
    return pdv_similarity


def load_embedding_cosines(folder, n_rows, invalid):
    folder = Path(folder)
    files = sorted(folder.glob("*.npy"))
    if not files:
        raise FileNotFoundError(f"No .npy embedding files found in {folder.resolve()}")

    result = {}
    for f in files:
        name = f.stem.split(" ")[0]
        emb  = np.load(f)
        emb  = emb[~invalid]
        result[name] = cosine_similarity(emb).astype(np.float32)
        print(f"Computed cosine matrix: {name} {result[name].shape}")
    return result


# RDKit physicochemical descriptor vector (PDV) cosine similarity.
rdkit_pdv = compute_rdkit_pdv_cosine(biogen)

# Learned / supplied embedding cosine similarities.
cosines = load_embedding_cosines(EMBEDDING_DIR, len(biogen), invalid)

# Combine fingerprint, RDKit PDV and embedding similarity matrices,
# then remove MiniMol and MolE from displayed analyses.
similarity_matrices_all = {
    **tan_mats,
    "RDKit PDV": rdkit_pdv,
    **cosines,
}
similarity_matrices = {
    name: matrix
    for name, matrix in similarity_matrices_all.items()
    if not is_excluded_representation(name)
}

removed = [name for name in similarity_matrices_all if is_excluded_representation(name)]
print("Excluded MiniMol/MolE representations:", removed if removed else "none found")
print("Displayed representations:", list(similarity_matrices))


Computed RDKit PDV cosine matrix: (3515, 3515) from 203 retained descriptors
Computed cosine matrix: COATI (3515, 3515)
Computed cosine matrix: CheMeleon (3515, 3515)
Computed cosine matrix: ChemBERTa (3515, 3515)
Computed cosine matrix: MiniMol (3515, 3515)
Computed cosine matrix: MolBERT (3515, 3515)
Computed cosine matrix: MolCLR (3515, 3515)
Computed cosine matrix: MolE (3515, 3515)
Computed cosine matrix: MolFormer (3515, 3515)
Computed cosine matrix: UniMol (3515, 3515)
Excluded MiniMol/MolE representations: ['MiniMol', 'MolE']
Displayed representations: ['ECFP4', 'Atom Pair', 'Torsion', 'RDKit PDV', 'COATI', 'CheMeleon', 'ChemBERTa', 'MolBERT', 'MolCLR', 'MolFormer', 'UniMol']


## Fixed top-pair calculations

In [19]:
def _top_k_indices_desc(values, k, rng):
    """ Indices of exactly k largest finite values; random tie-break only."""
    values     = np.asarray(values, dtype=float)
    finite_idx = np.flatnonzero(np.isfinite(values))
    if len(finite_idx) == 0:
        return np.array([], dtype=int)

    k = min(int(k), len(finite_idx))
    finite_values = values[finite_idx]
    tie   = rng.random(len(finite_idx))
    order = np.lexsort((tie, finite_values))
    return finite_idx[order[-k:]]


def compute_fixed_top_pair_summary(
    similarity_matrices,
    df,
    endpoints,
    pair_counts=(100_000, 10_000, 1_000),
    random_state=42,
):
    """
    For each endpoint × representation, select exactly the top-k most-similar
    valid molecular pairs and summarize |Δendpoint|.

    Pair IDs are returned in original dataframe row coordinates so pair sets
    can be compared across representations with Jaccard similarity.
    """
    rng = np.random.default_rng(random_state)
    rows = []
    pair_sets = {endpoint: {} for endpoint in endpoints}

    for endpoint in endpoints:
        vals = pd.to_numeric(df[endpoint], errors="coerce").to_numpy(float)
        valid_idx = np.flatnonzero(np.isfinite(vals))
        y = vals[valid_idx]
        i, j = np.triu_indices(len(y), k=1)
        absdiff = np.abs(y[i] - y[j])
        orig_i, orig_j = valid_idx[i], valid_idx[j]

        for rep, matrix in similarity_matrices.items():
            matrix = np.asarray(matrix, dtype=float)

            sim             = matrix[np.ix_(valid_idx, valid_idx)][i, j]
            valid_pair_mask = np.isfinite(sim) & np.isfinite(absdiff)
            sim_v  = sim[valid_pair_mask]
            diff_v = absdiff[valid_pair_mask]
            oi_v   = orig_i[valid_pair_mask]
            oj_v   = orig_j[valid_pair_mask]

            pair_sets[endpoint][rep] = {}

            for k_requested in pair_counts:
                selected = _top_k_indices_desc(sim_v, k_requested, rng)
                actual_k = len(selected)
                if actual_k == 0:
                    continue

                selected_pairs = {
                    (int(min(a, b)), int(max(a, b)))
                    for a, b in zip(oi_v[selected], oj_v[selected])
                }
                pair_sets[endpoint][rep][int(k_requested)] = selected_pairs

                s = sim_v[selected]
                d = diff_v[selected]
                rows.append({
                    "endpoint": endpoint,
                    "representation": rep,
                    "display_name": clean_representation_name(rep),
                    "top_pairs_requested": int(k_requested),
                    "n_top_pairs": actual_k,
                    "median_absdiff": float(np.median(d)),
                    "mean_absdiff": float(np.mean(d)),
                    "similarity_min": float(np.min(s)),
                    "similarity_median": float(np.median(s)),
                    "similarity_max": float(np.max(s)),
                })

    return pd.DataFrame(rows), pair_sets


pair_summary_df, top_pair_sets = compute_fixed_top_pair_summary(
    similarity_matrices,
    biogen,
    ENDPOINTS,
    pair_counts=TOP_PAIR_COUNTS,
    random_state=RANDOM_STATE,
)

pair_summary_df.head(12)

,endpoint,representation,display_name,top_pairs_requested,n_top_pairs,median_absdiff,mean_absdiff,similarity_min,similarity_median,similarity_max
0,LOG HLM_CLint (mL/min/kg),ECFP4,ECFP4,100000,100000,0.562487,0.661457,0.238095,0.260274,0.978261
1,LOG HLM_CLint (mL/min/kg),ECFP4,ECFP4,10000,10000,0.498071,0.605857,0.322581,0.361702,0.978261
2,LOG HLM_CLint (mL/min/kg),ECFP4,ECFP4,1000,1000,0.368954,0.482846,0.500000,0.580000,0.978261
3,LOG HLM_CLint (mL/min/kg),Atom Pair,Atom Pair,100000,100000,0.606579,0.697126,0.447479,0.469758,1.000000
4,LOG HLM_CLint (mL/min/kg),Atom Pair,Atom Pair,10000,10000,0.512841,0.612781,0.517241,0.538462,1.000000
5,LOG HLM_CLint (mL/min/kg),Atom Pair,Atom Pair,1000,1000,0.332255,0.440845,0.610236,0.658274,1.000000
6,LOG HLM_CLint (mL/min/kg),Torsion,Torsion,100000,100000,0.593518,0.679251,0.264706,0.294118,1.000000
7,LOG HLM_CLint (mL/min/kg),Torsion,Torsion,10000,10000,0.527393,0.617528,0.368421,0.411765,1.000000
8,LOG HLM_CLint (mL/min/kg),Torsion,Torsion,1000,1000,0.365643,0.495881,0.555556,0.627679,1.000000
9,LOG HLM_CLint (mL/min/kg),RDKit PDV,RDKit PDV,100000,100000,0.477043,0.573798,0.491007,0.556959,0.999216


## Jaccard similarity of the top-1k pair sets

In [20]:
def jaccard_similarity(set_a, set_b):
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else np.nan


def compute_jaccard_matrices(pair_sets, endpoints, reps, k=1_000):
    result = {}
    for endpoint in endpoints:
        J = np.full((len(reps), len(reps)), np.nan, dtype=float)
        for a, rep_a in enumerate(reps):
            sa = pair_sets[endpoint].get(rep_a, {}).get(k, set())
            for b, rep_b in enumerate(reps):
                sb = pair_sets[endpoint].get(rep_b, {}).get(k, set())
                J[a, b] = jaccard_similarity(sa, sb)
        result[endpoint] = J
    return result


rep_names  = list(similarity_matrices)
rep_labels = [clean_representation_name(r) for r in rep_names]
jaccard_top1k = compute_jaccard_matrices(
    top_pair_sets,
    ENDPOINTS,
    rep_names,
    k=1_000,
)

# Long-form table is convenient for export / downstream statistics.
jaccard_rows = []
for endpoint, J in jaccard_top1k.items():
    for i, a in enumerate(rep_names):
        for j, b in enumerate(rep_names):
            jaccard_rows.append({
                "endpoint": endpoint,
                "representation_a": a,
                "representation_b": b,
                "jaccard_top_1k": J[i, j],
            })
jaccard_top1k_df = pd.DataFrame(jaccard_rows)
jaccard_top1k_df.to_csv(OUTPUT_DIR / "top_1k_pair_jaccard.csv", index=False)

jaccard_top1k_df.head()

,endpoint,representation_a,representation_b,jaccard_top_1k
0,LOG HLM_CLint (mL/min/kg),ECFP4,ECFP4,1.000000
1,LOG HLM_CLint (mL/min/kg),ECFP4,Atom Pair,0.352265
2,LOG HLM_CLint (mL/min/kg),ECFP4,Torsion,0.557632
3,LOG HLM_CLint (mL/min/kg),ECFP4,RDKit PDV,0.178550
4,LOG HLM_CLint (mL/min/kg),ECFP4,COATI,0.343183


## Combined figure: publication styling from the reference

Four dumbbell panels (a–d) are followed by four top-1k Jaccard panels (e–h), using the same blue-grey/teal palette, Arial typography, spacing, panel labels, grey diagonal and colour-bar treatment as the supplied reference.


In [ ]:
# ============================================================
# Font sizes / marker sizes
# ============================================================

# Dumbbell panels a-d
DUMBBELL_Y_TICK_SIZE = 26
DUMBBELL_X_TICK_SIZE = 24


DUMBBELL_AXIS_TITLE_SIZE = 30

# Dumbbell circles
DUMBBELL_MARKER_SIZE = 22

DUMBBELL_CONNECTOR_WIDTH = 3.5


# Jaccard panels e-h
# Match representation-name size to dumbbell panels
JACCARD_TICK_SIZE = DUMBBELL_Y_TICK_SIZE
JACCARD_VALUE_SIZE = 17


# Dumbbell legend
# Larger legend title + large legend entries
LEGEND_TEXT_SIZE = 30
LEGEND_TITLE_SIZE = 34


# Endpoint titles / panel labels
SUBPLOT_TITLE_SIZE = 32
PANEL_LABEL_SIZE = 32


# Jaccard colourbar
COLORBAR_TICK_SIZE = 24
COLORBAR_TITLE_SIZE = 29

# Colourbar width
COLORBAR_THICKNESS = 40

# Distance between top of colourbar and title
COLORBAR_TITLE_GAP = 0.018


GLOBAL_FONT_SIZE = 19

# ============================================================
#  Styling
# ============================================================

PAIR_STYLE = {
    100_000: {
        "label": "Top 100k pairs",
        "color": "#687780",
    },
    10_000: {
        "label": "Top 10k pairs",
        "color": "#4C8DA3",
    },
    1_000: {
        "label": "Top 1k pairs",
        "color": "#0B5573",
    },
}

TOP_PAIR_COUNTS = (
    100_000,
    10_000,
    1_000,
)

CONNECTOR_COLOUR = "#A7B0B5"

JACCARD_DIAGONAL_COLOUR = "#D9DDE0"

JACCARD_COLOURSCALE = [
    [0.00, "#F2F5F6"],
    [0.33, "#A8C5CE"],
    [0.66, "#4C8DA3"],
    [1.00, "#0B5573"],
]

# Black borders around visible Jaccard cells
JACCARD_BORDER_COLOUR = "#000000"
JACCARD_BORDER_WIDTH = 1.2


ENDPOINT_LABELS = {
    "LOG HLM_CLint (mL/min/kg)":
        "Human liver microsomal clearance",

    "LOG MDR1-MDCK ER (B-A/A-B)":
        "MDR1–MDCK efflux ratio",

    "LOG SOLUBILITY PH 6.8 (ug/mL)":
        "Solubility at pH 6.8",

    "LOG RLM_CLint (mL/min/kg)":
        "Rat liver microsomal clearance",
}


# ============================================================
# Helpers
# ============================================================

def _axis_layout_name(axis_letter, axis_number):
    """ Return Plotly layout axis name (xaxis, xaxis2, ...)."""

    return (
        f"{axis_letter}axis"
        if axis_number == 1
        else f"{axis_letter}axis{axis_number}"
    )


def _plotly_marker_style(
    k,
    size=DUMBBELL_MARKER_SIZE,
):
    """ Marker styling for the three fixed top-pair sets."""

    color = PAIR_STYLE[k]["color"]

    if k == 100_000:
        return {
            "symbol": "circle-open",
            "size": size,
            "color": color,
            "line": {
                "color": color,
                "width": 2.5,
            },
        }

    return {
        "symbol": "circle",
        "size": size,
        "color": color,
        "line": {
            "color": "white",
            "width": 1.0,
        },
    }


# ============================================================
# Combined figure
# ============================================================

def plot_combined_publication_style(
    summary_df,
    jaccard_matrices,
    endpoints,
    rep_labels,
    width=2500,
    height=2800,
    show_overlap_values=True,
    lower_triangle_only=True,
    output_stem=None,
):
    """
    Combined dumbbell + Jaccard figure.

    Panels a-d
    ----------
    Median absolute endpoint difference for:
        - top 100k most-similar molecular pairs
        - top 10k most-similar molecular pairs
        - top 1k most-similar molecular pairs

    Panels e-h
    ----------
    Jaccard overlap of the top-1k molecular-pair sets.
    """

    # ========================================================
    # Subplot titles
    # ========================================================

    subplot_titles = (
        [
            ENDPOINT_LABELS.get(
                endpoint,
                endpoint,
            )
            for endpoint in endpoints
        ]
        +
        [
            ENDPOINT_LABELS.get(
                endpoint,
                endpoint,
            )
            for endpoint in endpoints
        ]
    )

    # ========================================================
    # Figure structure
    #
    #       a           b
    #       c           d
    #       e           f
    #       g           h
    #
    # a-d = dumbbells
    # e-h = Jaccard matrices
    # ========================================================

    fig = make_subplots(
        rows=4,
        cols=2,

        subplot_titles=subplot_titles,

        row_heights=[
            0.28,
            0.28,
            0.22,
            0.22,
        ],

        horizontal_spacing=0.10,

        vertical_spacing=0.075,
    )

    # ========================================================
    # Panels a-d: fixed top-pair dumbbells
    # ========================================================

    for panel_index, endpoint in enumerate(endpoints):

        row = panel_index // 2 + 1
        col = panel_index % 2 + 1
        panel_number = panel_index + 1

        # ----------------------------------------------------
        # Endpoint-specific data
        # ----------------------------------------------------

        sub = summary_df[
            summary_df["endpoint"].eq(endpoint)
        ].copy()

        pivot = sub.pivot(
            index="display_name",
            columns="top_pairs_requested",
            values="median_absdiff",
        )

        available = [
            k
            for k in TOP_PAIR_COUNTS
            if k in pivot.columns
        ]

        if not available:
            continue

        # Sort representations by top-1k performance
        order_key = (
            1_000
            if 1_000 in pivot.columns
            else available[-1]
        )

        pivot = pivot.sort_values(
            order_key,
            ascending=True,
        )

        representation_order = (
            pivot.index.tolist()
        )

        # ----------------------------------------------------
        # Connector lines
        # ----------------------------------------------------

        for display_name in representation_order:

            xs = [
                float(
                    pivot.loc[
                        display_name,
                        k,
                    ]
                )
                for k in available
                if pd.notna(
                    pivot.loc[
                        display_name,
                        k,
                    ]
                )
            ]

            if len(xs) >= 2:

                fig.add_trace(
                    go.Scatter(
                        x=[
                            min(xs),
                            max(xs),
                        ],

                        y=[
                            display_name,
                            display_name,
                        ],

                        mode="lines",

                        line={
                            "color":
                                CONNECTOR_COLOUR,

                            "width":
                                DUMBBELL_CONNECTOR_WIDTH,
                        },

                        showlegend=False,

                        hoverinfo="skip",
                    ),

                    row=row,
                    col=col,
                )

        # ----------------------------------------------------
        # Pair-count markers
        # ----------------------------------------------------

        for k in available:

            values = (
                pivot[k]
                .to_numpy(float)
            )

            labels = (
                pivot.index
                .to_numpy(object)
            )

            metadata = (
                sub[
                    sub[
                        "top_pairs_requested"
                    ].eq(k)
                ]
                .set_index(
                    "display_name"
                )
            )

            customdata = np.array(
                [
                    [
                        int(
                            metadata.loc[
                                name,
                                "n_top_pairs",
                            ]
                        ),

                        float(
                            metadata.loc[
                                name,
                                "similarity_min",
                            ]
                        ),

                        float(
                            metadata.loc[
                                name,
                                "similarity_median",
                            ]
                        ),

                        float(
                            metadata.loc[
                                name,
                                "similarity_max",
                            ]
                        ),
                    ]

                    for name in labels
                ],

                dtype=float,
            )

            fig.add_trace(
                go.Scatter(
                    x=values,
                    y=labels,

                    mode="markers",

                    name=(
                        PAIR_STYLE[k]["label"]
                    ),

                    legendgroup=(
                        f"pair_count_{k}"
                    ),

                    # Only panel b contributes
                    # visible legend entries
                    showlegend=(
                        panel_number == 2
                    ),

                    marker=(
                        _plotly_marker_style(
                            k,
                            size=DUMBBELL_MARKER_SIZE,
                        )
                    ),

                    customdata=customdata,

                    hovertemplate=(
                        "<b>%{y}</b><br>"
                        + PAIR_STYLE[k]["label"]
                        + "<br>"
                        "Median |Δ endpoint|: "
                        "%{x:.3f} log₁₀ units<br>"
                        "Pairs retained: "
                        "%{customdata[0]:,.0f}<br>"
                        "Minimum similarity: "
                        "%{customdata[1]:.3f}<br>"
                        "Median similarity: "
                        "%{customdata[2]:.3f}<br>"
                        "Maximum similarity: "
                        "%{customdata[3]:.3f}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=col,
            )

        # ----------------------------------------------------
        # Representation ordering
        # ----------------------------------------------------

        fig.update_yaxes(
            type="category",

            categoryorder="array",

            categoryarray=(
                representation_order
            ),

            autorange="reversed",

            row=row,
            col=col,
        )

    # ========================================================
    # Panels e-h: top-1k Jaccard overlap
    # ========================================================

    for panel_index, endpoint in enumerate(endpoints):

        row = panel_index // 2 + 3
        col = panel_index % 2 + 1

        J = np.asarray(
            jaccard_matrices[endpoint],
            dtype=float,
        ).copy()

        expected_shape = (
            len(rep_labels),
            len(rep_labels),
        )

        if J.shape != expected_shape:

            raise ValueError(
                f"{endpoint}: Jaccard matrix shape "
                f"{J.shape}; expected "
                f"{expected_shape}."
            )

        n = len(rep_labels)

        diagonal_mask = np.eye(
            n,
            dtype=bool,
        )

        overlap_z = J.copy()

        # ----------------------------------------------------
        # Lower triangle only
        # ----------------------------------------------------

        if lower_triangle_only:

            overlap_z[
                np.triu_indices(
                    n,
                    k=1,
                )
            ] = np.nan

        # Remove diagonal from coloured layer
        overlap_z[
            diagonal_mask
        ] = np.nan

        # ----------------------------------------------------
        # Neutral grey diagonal
        # ----------------------------------------------------

        diagonal_z = np.full_like(
            J,
            np.nan,
            dtype=float,
        )

        diagonal_z[
            diagonal_mask
        ] = 1.0

        # ----------------------------------------------------
        # Heatmap values
        # ----------------------------------------------------

        if show_overlap_values:

            text = np.full(
                J.shape,
                "",
                dtype=object,
            )

            finite = np.isfinite(
                overlap_z
            )

            text[finite] = [
                f"{v:.2f}"
                for v in overlap_z[finite]
            ]

            texttemplate = "%{text}"

        else:

            text = None
            texttemplate = None

        # ----------------------------------------------------
        # Jaccard colour layer
        # ----------------------------------------------------

        fig.add_trace(
            go.Heatmap(
                x=rep_labels,
                y=rep_labels,
                z=overlap_z,

                zmin=0,
                zmax=1,

                coloraxis="coloraxis",

                text=text,

                texttemplate=(
                    texttemplate
                ),

                textfont={
                    "family": "Arial",
                    "size":
                        JACCARD_VALUE_SIZE,
                },

                hoverongaps=False,

                hovertemplate=(
                    "<b>%{y}</b> vs "
                    "<b>%{x}</b><br>"
                    "Top-1k pair-set "
                    "Jaccard overlap: "
                    "%{z:.3f}"
                    "<extra></extra>"
                ),

                xgap=0,
                ygap=0,

                showscale=False,
            ),

            row=row,
            col=col,
        )

        # ----------------------------------------------------
        # Neutral grey self-overlap diagonal
        # ----------------------------------------------------

        fig.add_trace(
            go.Heatmap(
                x=rep_labels,
                y=rep_labels,
                z=diagonal_z,

                zmin=0,
                zmax=1,

                colorscale=[
                    [
                        0.0,
                        JACCARD_DIAGONAL_COLOUR,
                    ],
                    [
                        1.0,
                        JACCARD_DIAGONAL_COLOUR,
                    ],
                ],

                showscale=False,

                text=None,
                texttemplate=None,

                hoverinfo="skip",
                hoverongaps=False,

                xgap=0,
                ygap=0,
            ),

            row=row,
            col=col,
        )

        # ----------------------------------------------------
        # Black outlines around visible Jaccard cells
        # ----------------------------------------------------

        for i in range(n):

            for j in range(n):

                if (
                    lower_triangle_only
                    and j > i
                ):
                    continue

                fig.add_shape(
                    type="rect",

                    x0=j - 0.5,
                    x1=j + 0.5,

                    y0=i - 0.5,
                    y1=i + 0.5,

                    line={
                        "color":
                            JACCARD_BORDER_COLOUR,

                        "width":
                            JACCARD_BORDER_WIDTH,
                    },

                    fillcolor=(
                        "rgba(0,0,0,0)"
                    ),

                    layer="above",

                    row=row,
                    col=col,
                )

        fig.update_yaxes(
            autorange="reversed",

            row=row,
            col=col,
        )

    # ========================================================
    # Axis styling: dumbbell panels a-d
    # ========================================================

    for panel_number in range(
        1,
        len(endpoints) + 1,
    ):

        xaxis_name = (
            _axis_layout_name(
                "x",
                panel_number,
            )
        )

        yaxis_name = (
            _axis_layout_name(
                "y",
                panel_number,
            )
        )

        # Panels c and d are the bottom dumbbell row
        is_bottom_dumbbell_row = (
            panel_number > 2
        )

        # ----------------------------------------------------
        # Dumbbell x-axis
        # ----------------------------------------------------

        fig.layout[
            xaxis_name
        ].update(

            title_text=(
                "Median absolute endpoint "
                "difference (log₁₀ units)"

                if is_bottom_dumbbell_row

                else None
            ),

            rangemode="tozero",

            automargin=True,

            ticks="outside",

            ticklen=9,
            tickwidth=1.6,
            tickcolor="#333333",

            showline=True,
            linewidth=1.2,
            linecolor="#333333",

            # ------------------------------------------------
            # Grid lines OFF
            # ------------------------------------------------
            showgrid=False,

            zeroline=False,

            tickfont={
                "family": "Arial",
                "size":
                    DUMBBELL_X_TICK_SIZE,
                "color": "#202124",
            },

            # Larger title under panels c/d
            title_font={
                "family": "Arial",
                "size":
                    DUMBBELL_AXIS_TITLE_SIZE,
                "color": "#202124",
            },
        )

        # ----------------------------------------------------
        # Dumbbell y-axis
        # ----------------------------------------------------

        fig.layout[
            yaxis_name
        ].update(

            automargin=True,

            ticks="",

            showline=False,
            showgrid=False,

            tickfont={
                "family": "Arial",
                "size":
                    DUMBBELL_Y_TICK_SIZE,
                "color": "#202124",
            },
        )

    # ========================================================
    # Axis styling: Jaccard panels e-h
    # ========================================================

    for panel_number in range(
        len(endpoints) + 1,
        2 * len(endpoints) + 1,
    ):

        xaxis_name = (
            _axis_layout_name(
                "x",
                panel_number,
            )
        )

        yaxis_name = (
            _axis_layout_name(
                "y",
                panel_number,
            )
        )

        # ----------------------------------------------------
        # Jaccard x-axis
        # ----------------------------------------------------

        fig.layout[
            xaxis_name
        ].update(

            automargin=True,

            tickangle=-45,

            ticks="",

            ticklabelstandoff=8,

            showline=False,
            showgrid=False,

            tickfont={
                "family": "Arial",
                "size":
                    JACCARD_TICK_SIZE,
                "color": "#202124",
            },
        )

        # ----------------------------------------------------
        # Jaccard y-axis
        # ----------------------------------------------------

        fig.layout[
            yaxis_name
        ].update(

            automargin=True,

            ticks="",

            ticklabelstandoff=16,

            showline=False,
            showgrid=False,

            tickfont={
                "family": "Arial",
                "size":
                    JACCARD_TICK_SIZE,
                "color": "#202124",
            },
        )

    # ========================================================
    # Single enlarged dumbbell legend beside panel b
    # ========================================================

    top_right_dumbbell_x_domain = (
        fig.layout[
            _axis_layout_name(
                "x",
                2,
            )
        ].domain
    )

    top_right_dumbbell_y_domain = (
        fig.layout[
            _axis_layout_name(
                "y",
                2,
            )
        ].domain
    )

    fig.update_layout(
        legend={
            "orientation": "v",

            "x": (
                top_right_dumbbell_x_domain[1]
                - 0.025
            ),

            "xanchor": "left",

            "y": (
                top_right_dumbbell_y_domain[1]
            ),

            "yanchor": "top",

            "title": {
                "text":
                    "Most-similar pairs",

                "font": {
                    "family": "Arial",
                    "size":
                        LEGEND_TITLE_SIZE,
                    "color": "#202124",
                },
            },

            # Large Top 100k / Top 10k / Top 1k text
            "font": {
                "family": "Arial",
                "size":
                    LEGEND_TEXT_SIZE,
                "color": "#202124",
            },

            "bgcolor":
                "rgba(255,255,255,0)",

            "borderwidth": 0,

            # Scale legend circles with actual markers
            "itemsizing": "trace",

            "itemwidth": 50,

            "tracegroupgap": 12,
        }
    )

    # ========================================================
    # Shared enlarged Jaccard colourbar
    # ========================================================

    # Panel e y-domain
    top_heatmap_y_domain = (
        fig.layout[
            _axis_layout_name(
                "y",
                5,
            )
        ].domain
    )

    # Panel f x-domain
    top_right_heatmap_x_domain = (
        fig.layout[
            _axis_layout_name(
                "x",
                6,
            )
        ].domain
    )

    heatmap_y_bottom = float(
        top_heatmap_y_domain[0]
    )

    heatmap_y_top = float(
        top_heatmap_y_domain[1]
    )

    heatmap_height = (
        heatmap_y_top
        - heatmap_y_bottom
    )

    heatmap_y_centre = (
        heatmap_y_bottom
        + heatmap_y_top
    ) / 2

    # --------------------------------------------------------
    # Explicit colourbar position
    # --------------------------------------------------------

    colorbar_x = (
        float(
            top_right_heatmap_x_domain[1]
        )
        + 0.018
    )

    # Approximate centre of colourbar
    colorbar_title_x = (
        colorbar_x
        + (
            COLORBAR_THICKNESS
            / width
        ) / 2
    )

    colorbar_title_y = (
        heatmap_y_top
        + COLORBAR_TITLE_GAP
    )

    fig.update_layout(
        coloraxis={

            "colorscale":
                JACCARD_COLOURSCALE,

            "cmin":
                0,

            "cmax":
                1,

            "colorbar": {

                "x":
                    colorbar_x,

                "xanchor":
                    "left",

                "y":
                    heatmap_y_centre,

                "yanchor":
                    "middle",

                "len":
                    heatmap_height,

                "lenmode":
                    "fraction",

                "thickness":
                    COLORBAR_THICKNESS,

                "thicknessmode":
                    "pixels",

                "tickmode":
                    "array",

                "tickvals": [
                    0,
                    0.5,
                    1,
                ],

                "ticktext": [
                    "0",
                    "0.5",
                    "1",
                ],

                "ticks":
                    "outside",

                "ticklen":
                    11,

                "tickwidth":
                    1.6,

                "tickfont": {
                    "family": "Arial",
                    "size":
                        COLORBAR_TICK_SIZE,
                    "color": "#202124",
                },

                # Separate annotation used for title
                "title": {
                    "text": "",
                },

                "outlinewidth":
                    0,
            },
        }
    )

    # --------------------------------------------------------
    # Separate Jaccard colourbar title
    # --------------------------------------------------------

    fig.add_annotation(
        x=colorbar_title_x,
        y=colorbar_title_y,

        xref="paper",
        yref="paper",

        text="Jaccard overlap",

        showarrow=False,

        xanchor="center",
        yanchor="bottom",

        font={
            "family": "Arial",
            "size":
                COLORBAR_TITLE_SIZE,
            "color": "#202124",
        },
    )

    # ========================================================
    # Endpoint subplot titles
    # ========================================================

    for annotation in (
        fig.layout.annotations[
            : 2 * len(endpoints)
        ]
    ):

        annotation.update(
            font={
                "family": "Arial",
                "size":
                    SUBPLOT_TITLE_SIZE,
                "color": "#202124",
            },

            yshift=9,
        )

    # ========================================================
    # Ordered panel labels a-h
    # ========================================================

    for panel_index in range(
        2 * len(endpoints)
    ):

        axis_number = (
            panel_index + 1
        )

        xref = (
            "x domain"
            if axis_number == 1
            else f"x{axis_number} domain"
        )

        yref = (
            "y domain"
            if axis_number == 1
            else f"y{axis_number} domain"
        )

        if panel_index < len(endpoints):

            letter_x = -0.18
            letter_y = 1.13

        else:

            letter_x = -0.16
            letter_y = 1.13

        fig.add_annotation(

            x=letter_x,
            y=letter_y,

            xref=xref,
            yref=yref,

            text=(
                f"<b>"
                f"{chr(ord('a') + panel_index)}"
                f"</b>"
            ),

            showarrow=False,

            xanchor="left",
            yanchor="top",

            font={
                "family": "Arial",
                "size":
                    PANEL_LABEL_SIZE,
                "color": "#111111",
            },
        )

    # ========================================================
    # Final layout
    # ========================================================

    fig.update_layout(

        width=width,
        height=height,

        paper_bgcolor="white",
        plot_bgcolor="white",

        font={
            "family": "Arial",
            "size":
                GLOBAL_FONT_SIZE,
            "color": "#202124",
        },

        margin={
            "l": 300,
            "r": 420,
            "t": 145,
            "b": 260,
        },

        hoverlabel={
            "bgcolor": "white",
            "bordercolor": "#687780",

            "font": {
                "family": "Arial",
                "size": 18,
                "color": "#202124",
            },
        },
    )

    # ========================================================
    # Optional export
    # ========================================================

    if output_stem is not None:

        output_stem = Path(
            output_stem
        )

        output_stem.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        # ----------------------------------------------------
        # Interactive HTML
        # ----------------------------------------------------

        fig.write_html(
            str(
                output_stem.with_suffix(
                    ".html"
                )
            )
        )

        for suffix in (
            ".png",
            ".pdf",
        ):

            try:

                fig.write_image(
                    str(
                        output_stem.with_suffix(
                            suffix
                        )
                    ),

                    width=width,
                    height=height,
                )

            except Exception as exc:

                print(
                    f"Static {suffix} export skipped "
                    f"({exc})."
                )

                break

    return fig


# ============================================================
# Generate and display
# ============================================================

fig_combined = plot_combined_publication_style(
    summary_df=pair_summary_df,
    jaccard_matrices=jaccard_top1k,
    endpoints=ENDPOINTS,
    rep_labels=rep_labels,

    width=2500,
    height=2800,

    show_overlap_values=True,
    lower_triangle_only=True,

    output_stem=(
        OUTPUT_DIR
        / "fixed_pairs_plus_top1k_jaccard"
    ),
)

fig_combined.show()

## Export analysis tables

In [ ]:
pair_summary_df.to_csv(OUTPUT_DIR / "fixed_top_pair_summary.csv", index=False)

print("Wrote:")
for path in sorted(OUTPUT_DIR.glob("fixed_top_pair*")):
    print(" -", path)
for path in sorted(OUTPUT_DIR.glob("top_1k_pair_jaccard.csv")):
    print(" -", path)

Wrote:
 - qsar_figures/fixed_top_pair_summary.csv
 - qsar_figures/top_1k_pair_jaccard.csv
